# Sales Data Analysis

This notebook demonstrates how to connect to the PostgreSQL data warehouse and perform analysis on the sales data.

In [ ]:
# Import required libraries
import psycopg2
import pandas as pd
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Connect to database
conn = psycopg2.connect(
    host=os.getenv('DB_HOST', 'localhost'),
    port=os.getenv('DB_PORT', '5432'),
    database=os.getenv('DB_NAME', 'sales_dwh'),
    user=os.getenv('DB_USER', 'etl_user'),
    password=os.getenv('DB_PASSWORD', 'etl_password')
)

print("Connected to database successfully!")

## Sales by Category

In [ ]:
# Query sales by category
query = """
SELECT 
    p.category,
    COUNT(*) as order_count,
    SUM(f.quantity) as total_quantity,
    ROUND(SUM(f.total_amount), 2) as total_revenue
FROM dwh.fact_sales f
JOIN dwh.dim_product p ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

df_category = pd.read_sql_query(query, conn)
df_category

## Sales by Country

In [ ]:
# Query sales by country
query = """
SELECT 
    c.country,
    COUNT(DISTINCT c.customer_key) as customer_count,
    ROUND(SUM(f.total_amount), 2) as total_revenue
FROM dwh.fact_sales f
JOIN dwh.dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.country
ORDER BY total_revenue DESC;
"""

df_country = pd.read_sql_query(query, conn)
df_country

## Monthly Sales Trend

In [ ]:
# Query monthly sales trend
query = """
SELECT 
    d.year,
    d.month,
    d.month_name,
    COUNT(*) as order_count,
    ROUND(SUM(f.total_amount), 2) as total_revenue
FROM dwh.fact_sales f
JOIN dwh.dim_date d ON f.date_key = d.date_id
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month;
"""

df_monthly = pd.read_sql_query(query, conn)
df_monthly

## Close Connection

In [ ]:
# Close database connection
conn.close()
print("Connection closed")